<a href="https://colab.research.google.com/github/carneiro-santos/dataton_fase_5_/blob/main/Analise_explorat%C3%B3ria_e_tratativa_dos_dados_Datathon_Fase_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import plotly.graph_objs as go
import plotly.express as px
import plotly.io as pio
import plotly.offline as pyo
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [42]:
df = pd.read_csv('PEDE_PASSOS_DATASET_FIAP.csv', sep=';', encoding='latin1')
df_2022 = pd.read_csv('df_2022.csv', sep=';', encoding='latin1')
df_2023 = pd.read_csv('df_2023.csv', sep=';', encoding='latin1')
df_2024 = pd.read_csv('df_2024.csv', sep=';', encoding='latin1')

In [43]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [44]:
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')

In [26]:
# Configurar pandas para exibir colunas inteiras
pd.set_option('display.max_colwidth', None)

In [45]:
# Configurar pandas para exibir colunas inteiras
pd.reset_option('display.max_colwidth', None)

In [46]:
df.to_csv('DF_PEDE_PASSOS_LIMPO.csv', sep=';', index=False)

In [47]:
def create_histogram(df, indicator, year, color):
    # Dynamically determine the correct column name
    # Check for the suffixed column name (e.g., INDE_2020)
    column_name_suffixed = f'{indicator}_{year}'
    # Check for the unsuffixed column name (e.g., INDE)
    column_name_unsuffixed = indicator

    if column_name_suffixed in df.columns:
        col_to_plot = column_name_suffixed
    elif column_name_unsuffixed in df.columns:
        col_to_plot = column_name_unsuffixed
    else:
        # If neither column name is found, skip this trace
        print(f"Warning: Column '{indicator}' not found for year {year} in DataFrame. Skipping histogram.")
        return None # Return None to indicate no trace should be added

    # Convert the column to numeric, replacing commas with dots and coercing errors
    df[col_to_plot] = pd.to_numeric(df[col_to_plot].astype(str).str.replace(',', '.'), errors='coerce')
    # Drop NaN values that resulted from coercion for plotting
    data_to_plot = df[col_to_plot].dropna()

    if data_to_plot.empty:
        print(f"Warning: No valid numeric data for '{col_to_plot}' after conversion. Skipping histogram.")
        return None

    return go.Histogram(
        x=data_to_plot,
        marker_color=color,
        opacity=0.75,
        xbins=dict(start=0, end=10.2, size=1),
        name=f'{indicator}_{year}' # Keep name consistent for legend
    )

In [49]:
# Definindo variáveis
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']
colors = ['#636EFA', '#EF553B', '#00CC96', '#800000', '#620080']
years = [ '2022', '2023', '2024']
dfs = [ df_2022, df_2023, df_2024]  # Listagem dos dataframes de cada ano

# Criando subplots com número de linhas baseado no número de indicadores e colunas baseado no número de anos
fig = make_subplots(rows=len(indicators), cols=len(years),
                    subplot_titles=[f'{indicator} {year}' for indicator in indicators for year in years])

# Iterar sobre cada indicador, ano e cor correspondente
for row_idx, indicator in enumerate(indicators):
    for col_idx, (year, df, color) in enumerate(zip(years, dfs, colors)):
        hist_trace = create_histogram(df, indicator, year, color)
        if hist_trace: # Only add the trace if it's not None
            fig.add_trace(hist_trace, row=row_idx + 1, col=col_idx + 1)

# Atualizando layout do gráfico
fig.update_layout(height=1200, width=1500, showlegend=False, bargap=0.1,
                  xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

In [53]:

import plotly.io as pio

# Garante que o gráfico será exibido (especialmente fora do Jupyter)
pio.renderers.default = "colab" # Changed from "browser" to "colab"

def calculate_mean_by_year(indicators_dict, indicators):
    means = {}
    for year, df in indicators_dict.items():
        current_year_indicators = []
        for indicator in indicators:
            suffixed_col = f'{indicator}_{year}'
            unsuffixed_col = indicator
            if suffixed_col in df.columns:
                current_year_indicators.append(suffixed_col)
            elif unsuffixed_col in df.columns:
                current_year_indicators.append(unsuffixed_col)
            else:
                current_year_indicators.append(None)

        yearly_means = []
        for indicator_col in current_year_indicators:
            if indicator_col and indicator_col in df.columns:
                # Ensure numeric conversion, handling commas as decimal points
                col_series = pd.to_numeric(df[indicator_col].astype(str).str.replace(',', '.'), errors='coerce')
                yearly_means.append(col_series.mean())
            else:
                yearly_means.append(np.nan)

        means[year] = yearly_means
    return means

# Função para plotar gráfico com Plotly
def plot_mean_indicators(means, indicators):
    fig = go.Figure()
    years = sorted(means.keys())

    for i, indicator in enumerate(indicators):
        y_values = []
        for year in years:
            value = means[year][i]
            y_values.append(value if pd.notna(value) else None)

        fig.add_trace(go.Scatter(
            x=years,
            y=y_values,
            mode='lines+markers',
            name=indicator,
            connectgaps=False
        ))

    fig.update_layout(
        title='Médias dos Indicadores de Desempenho por Ano (2022-2024)',
        xaxis_title='Ano',
        yaxis_title='Média dos Indicadores',
        template='plotly_white',
        legend_title="Indicadores",
        hovermode='x unified'
    )

    return fig

# Criando o dicionário de DataFrames para a função calculate_mean_by_year
indicators_dict = {
    '2022': df_2022,
    '2023': df_2023,
    '2024': df_2024
}

# Assegurando que 'indicators' está definido
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN'] # Redefinido para clareza, pois já existia

# Calculando as médias antes de plotar
means = calculate_mean_by_year(indicators_dict, indicators)

# ========= EXIBIÇÃO DO GRÁFICO =========
fig = plot_mean_indicators(means, indicators)
fig.show()

# Analise dos Tipos de Pedras


# 1.   Por ano


In [54]:
def load_data(file_path):
    return pd.read_csv(file_path, sep=';', encoding='latin1')

def preprocess_data(df, year):
    pedra_column_suffixed = f'PEDRA_{year}'
    pedra_column_unsuffixed = 'PEDRA'
    actual_pedra_column = None

    if pedra_column_suffixed in df.columns:
        actual_pedra_column = pedra_column_suffixed
    elif pedra_column_unsuffixed in df.columns:
        actual_pedra_column = pedra_column_unsuffixed
    else:
        # If no pedra column found, return an empty series
        print(f"Warning: No 'PEDRA' column found for year {year}. Skipping preprocessing.")
        return pd.Series(dtype=int) # Return an empty series

    df = df[[actual_pedra_column]].dropna()
    df_filtered = df[~df[actual_pedra_column].isin(['D9891/2A', '#NULO!'])]
    return df_filtered[actual_pedra_column].value_counts()

def create_dataframe_from_series(series, year):
    df = pd.DataFrame(series).reset_index()
    df.columns = ['Tipo de Pedra', 'Contagem']
    df['Ano'] = year
    return df

def setup_plot(df_combined, title, colors):
    fig = go.Figure()
    for pedra in df_combined['Tipo de Pedra'].unique():
        df_pedra = df_combined[df_combined['Tipo de Pedra'] == pedra]
        fig.add_trace(go.Bar(
            x=df_pedra['Ano'],
            y=df_pedra['Contagem'],
            name=pedra,
            marker_color=colors.get(pedra, '#000000'),  # Default black if not specified
            text=df_pedra['Contagem'],
            textposition='inside',
            insidetextfont=dict(color='black')  # Set the text color to white
        ))

    fig.update_layout(
        title=title,
        xaxis_title='Ano',
        yaxis_title='Número de Ocorrências',
        barmode='group',
        hovermode='x',
        legend=dict(orientation="h", yanchor="bottom", y=1.00, xanchor="center", x=0.5)

    )
    fig.show()

# Define the colors for each type of stone
cores_pedras = {
    'Quartzo': '#B9F2FF',  # Diamante
    'Ágata': '#FFC0CB',    # Light Pink
    'Ametista': '#9966CC', # Violet
    'Topázio': '#FFD700'   # Gold
}

# Load and preprocess the data

df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')

# Apply preprocessing and generate dataframes

pedra_counts_2022 = preprocess_data(df_2022, '2022')
pedra_counts_2023 = preprocess_data(df_2023, '2023')
pedra_counts_2024 = preprocess_data(df_2024, '2024')


df_2022 = create_dataframe_from_series(pedra_counts_2022, '2022')
df_2023 = create_dataframe_from_series(pedra_counts_2023, '2023')
df_2024 = create_dataframe_from_series(pedra_counts_2024, '2024')

# Concatenate dataframes
df_combined = pd.concat([df_2022, df_2023, df_2024])

# Setup and show the plot
setup_plot(df_combined, 'Comparação de Ocorrências de Pedras por Tipo em Cada Ano', cores_pedras)

# 2.   Genero


In [33]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Reload original dataframes to ensure they have the necessary columns
df_2020 = pd.read_csv('df_2020.csv', sep=';', encoding='latin1')
df_2021 = pd.read_csv('df_2021.csv', sep=';', encoding='latin1')
df_2022 = pd.read_csv('df_2022.csv', sep=';', encoding='latin1')
df_2023 = pd.read_csv('df_2023.csv', sep=';', encoding='latin1')
df_2024 = pd.read_csv('df_2024.csv', sep=';', encoding='latin1')

def get_pedra_col_name(df, year):
    if f'PEDRA_{year}' in df.columns:
        return f'PEDRA_{year}'
    elif 'PEDRA' in df.columns:
        return 'PEDRA'
    return None

def get_gender_col_name(df, year):
    possible_gender_cols = ['GÊNERO', 'Sexo', f'SEXO_ALUNO_{year}']
    for col in possible_gender_cols:
        if col in df.columns:
            return col
    return None

# Define common 'PEDRA' types for filtering
pedra_types = ['Ametista', 'Quartzo', 'Topázio', 'Ágata']

# Create a base DataFrame with all possible Sexo/PEDRA combinations for a complete merge
all_sexo = ['M', 'F']
all_pedra_types = ['Ametista', 'Quartzo', 'Topázio', 'Ágata']
base_comparison = pd.DataFrame([(s, p) for s in all_sexo for p in all_pedra_types], columns=['Sexo', 'PEDRA'])

years_to_process = ['2020', '2021', '2022', '2023', '2024']
dfs_to_process = [df_2020, df_2021, df_2022, df_2023, df_2024]

pedra_comparison = base_comparison.copy() # Start with the base combinations

for year, df_year in zip(years_to_process, dfs_to_process):
    pedra_col = get_pedra_col_name(df_year, year)
    gender_col = get_gender_col_name(df_year, year)

    if pedra_col and gender_col:
        df_clean = df_year.copy()

        # Standardize gender values
        df_clean['Sexo_Clean'] = df_clean[gender_col].astype(str).str.upper().str.strip()
        df_clean['Sexo_Clean'] = df_clean['Sexo_Clean'].map({
            'MASCULINO': 'M',
            'M': 'M',
            'FEMININO': 'F',
            'F': 'F'
        }).fillna(np.nan)

        # Filter for valid gender and pedra types
        df_clean = df_clean.dropna(subset=['Sexo_Clean'])
        df_clean = df_clean[df_clean[pedra_col].isin(pedra_types)]

        if not df_clean.empty:
            grouped_counts = df_clean.groupby(['Sexo_Clean', pedra_col]).size().reset_index(name=f'count_{year}')
            grouped_counts.rename(columns={'Sexo_Clean': 'Sexo', pedra_col: 'PEDRA'}, inplace=True)

            # Merge current year's counts into the main comparison DataFrame
            pedra_comparison = pd.merge(pedra_comparison, grouped_counts, on=['Sexo', 'PEDRA'], how='left')
        else:
            # If df_clean is empty, ensure the count column for this year is added with NaNs
            pedra_comparison[f'count_{year}'] = np.nan
    else:
        # If no pedra_col or gender_col found, add the column with NaNs
        pedra_comparison[f'count_{year}'] = np.nan

# After the loop, fill any remaining NaNs from left merges or missing years with 0
count_cols = [f'count_{y}' for y in years_to_process]
pedra_comparison[count_cols] = pedra_comparison[count_cols].fillna(0)

# Calculate total for all years for consistency
pedra_comparison['total'] = pedra_comparison[count_cols].sum(axis=1)

# Create a dataframe separated by gender, with the total of each stone
totais_feminino = pedra_comparison[pedra_comparison['Sexo'] == 'F'].groupby('PEDRA')['total'].sum().reset_index()
totais_masculino = pedra_comparison[pedra_comparison['Sexo'] == 'M'].groupby('PEDRA')['total'].sum().reset_index()

# Criando a figura para barras empilhadas por pedra e por gênero (sem separar por ano)
fig = go.Figure()

# Adicionando dados do total para o gênero feminino
fig.add_trace(go.Bar(
    x=totais_feminino['PEDRA'],
    y=totais_feminino['total'],
    name='Feminino',
    marker_color='#8B008B',
    hovertemplate='<b>Pedra:</b> %{x}<br><b>Sexo:</b> Feminino<br><b>Quantidade:</b> %{y}<extra></extra>'
))

# Adicionando dados do total para o gênero masculino
fig.add_trace(go.Bar(
    x=totais_masculino['PEDRA'],
    y=totais_masculino['total'],
    name='Masculino',
    marker_color='#0000CD',
    hovertemplate='<b>Pedra:</b> %{x}<br><b>Sexo:</b> Masculino<br><b>Quantidade:</b> %{y}<extra></extra>'
))

# Atualizando o layout para barras empilhadas
fig.update_layout(
    barmode='stack',
    title='Distribuição Total de Estudantes por Pedra e Gênero (2020-2024)',
    xaxis_title='Tipo de Pedra',
    yaxis_title='Número de Estudantes',
    template='plotly_white',
    legend_title='Gênero',
    #height=700
)
fig.show()

#  Media de Indicadores por Genero

In [34]:
import numpy as np
import pandas as pd

def calculate_mean_by_gender_per_year(df, year, indicators):
    df_copy = df.copy()

    # Dynamically determine the gender column name
    gender_col_name = None
    # Prioritize 'GÊNERO' as identified from column listings, then 'Sexo', then year-suffixed
    possible_gender_cols = ['GÊNERO', 'Sexo', f'SEXO_ALUNO_{year}']
    for col in possible_gender_cols:
        if col in df_copy.columns:
            gender_col_name = col
            break

    if not gender_col_name:
        # print(f"Warning: No valid gender column found for year {year}. Returning empty DataFrame for gender means.")
        empty_means = pd.DataFrame(index=['M', 'F'], columns=[f'{ind}_{year}' for ind in indicators])
        return empty_means

    # Standardize gender values to 'M' or 'F', converting other values to NaN
    df_copy['Sexo_Clean'] = df_copy[gender_col_name].astype(str).str.upper().str.strip()
    df_copy['Sexo_Clean'] = df_copy['Sexo_Clean'].map({
        'MASCULINO': 'M',
        'M': 'M',
        'FEMININO': 'F',
        'F': 'F'
    }).fillna(np.nan) # Any value not explicitly mapped becomes NaN

    # Drop rows where 'Sexo_Clean' is NaN
    df_cleaned = df_copy.dropna(subset=['Sexo_Clean'])

    # If no data left after cleaning, return an empty means DataFrame with 'M' and 'F' indices and NaNs
    if df_cleaned.empty:
        # print(f"Warning: No valid 'M' or 'F' gender data found in df for year {year} after cleaning '{gender_col_name}'. Returning empty DataFrame for gender means.")
        empty_means = pd.DataFrame(index=['M', 'F'], columns=[f'{ind}_{year}' for ind in indicators])
        return empty_means

    # Prepare list of actual column names to use for calculations
    actual_indicator_cols_for_grouping = []
    for indicator in indicators:
        suffixed_col = f'{indicator}_{year}' # e.g., INDE_2020
        unsuffixed_col = indicator          # e.g., INDE
        col_to_process = None

        # Prioritize unsuffixed name first, as per column inspection
        if unsuffixed_col in df_cleaned.columns:
            col_to_process = unsuffixed_col
        elif suffixed_col in df_cleaned.columns:
            col_to_process = suffixed_col

        if col_to_process:
            # Convert the column to numeric, coercing errors to NaN by replacing commas with dots
            df_cleaned[col_to_process] = pd.to_numeric(df_cleaned[col_to_process].astype(str).str.replace(',', '.'), errors='coerce')
            actual_indicator_cols_for_grouping.append(col_to_process)
        else:
            # print(f"Warning: Indicator column '{indicator}' (or '{suffixed_col}') not found for year {year}. Skipping.")
            pass # Keep silent for now to avoid too many warnings if it's expected behavior


    if not actual_indicator_cols_for_grouping:
        # print(f"Warning: No valid indicator columns found for year {year} after gender cleaning. Returning empty DataFrame for gender means.")
        empty_means = pd.DataFrame(index=['M', 'F'], columns=[f'{ind}_{year}' for ind in indicators])
        return empty_means

    # Group by 'Sexo_Clean' and calculate the mean for the actual indicator columns
    gender_means = df_cleaned.groupby('Sexo_Clean')[actual_indicator_cols_for_grouping].mean(numeric_only=True)

    # Rename columns to the consistent suffixed format (e.g., 'INDE_2020') for plotting
    rename_dict = {}
    for original_col in actual_indicator_cols_for_grouping:
        # If the column was 'INDE' and year is '2020', it should become 'INDE_2020'
        # If it was already 'INDE_2020', it remains 'INDE_2020'
        base_indicator = original_col.split('_')[0] if original_col.endswith(f'_{year}') else original_col
        rename_dict[original_col] = f'{base_indicator}_{year}'
    gender_means = gender_means.rename(columns=rename_dict)

    # Reindex to ensure 'M' and 'F' are always present, filling missing with NaN
    final_means = gender_means.reindex(['M', 'F'], fill_value=np.nan)

    # Ensure all original indicators are represented in the final DataFrame columns, with NaN if no data
    all_target_cols = [f'{ind}_{year}' for ind in indicators]

    # Reorder existing columns to match all_target_cols and add missing columns as NaN
    final_means = final_means.reindex(columns=all_target_cols)

    return final_means

In [35]:
# Indicadores a serem analisados
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']

# Calculando a média por sexo de cada dataframe
means_2020 = calculate_mean_by_gender_per_year(df_2020, '2020', indicators)
means_2021 = calculate_mean_by_gender_per_year(df_2021, '2021', indicators)
means_2022 = calculate_mean_by_gender_per_year(df_2022, '2022', indicators)
means_2023 = calculate_mean_by_gender_per_year(df_2023, '2023', indicators)
means_2024 = calculate_mean_by_gender_per_year(df_2024, '2024', indicators)



# Função para criar o gráfico interativo com menu para selecionar o indicador e texto abaixo da legenda
def plot_interactive_indicator_selector_with_text(means_2020, means_2021, means_2022, means_2023, means_2024, indicators):
    fig = go.Figure()

    # Adicionando os traces para cada indicador (inicialmente ocultos)
    for indicator in indicators:
        # Masculino
        fig.add_trace(go.Scatter(x=['2020', '2021', '2022', '2023', '2024'],
                                y=[means_2020.loc['M', f'{indicator}_2020'],
                                    means_2021.loc['M', f'{indicator}_2021'],
                                    means_2022.loc['M', f'{indicator}_2022'],
                                    means_2023.loc['M', f'{indicator}_2023'],
                                    means_2024.loc['M', f'{indicator}_2024']],
                                mode='lines+markers', name=f'{indicator} (Masculino)',
                                visible=False))

        # Feminino
        fig.add_trace(go.Scatter(x=['2020', '2021', '2022', '2023', '2024'],
                                y=[means_2020.loc['F', f'{indicator}_2020'],
                                    means_2021.loc['F', f'{indicator}_2021'],
                                    means_2022.loc['F', f'{indicator}_2022'],
                                    means_2023.loc['F', f'{indicator}_2023'],
                                    means_2024.loc['F', f'{indicator}_2024']],
                                mode='lines+markers', name=f'{indicator} (Feminino)',
                                visible=False))

    # Criando o menu interativo
    buttons = []
    for i, indicator in enumerate(indicators):
        buttons.append(dict(
            method='update',
            label=indicator,
            args=[{'visible': [False] * len(fig.data)},  # Oculta todos os traces
                {'title': f'Indicador: {indicator} (Masculino vs Feminino)'}]  # Atualiza o título
        ))

        # Mostra os traces correspondentes ao indicador selecionado
        buttons[i]['args'][0]['visible'][i * 2] = True  # Masculino
        buttons[i]['args'][0]['visible'][i * 2 + 1] = True  # Feminino

    # Adicionando o menu ao gráfico e estilizando o dropdown
    fig.update_layout(
        updatemenus=[{
            'buttons': buttons,
            'direction': 'down',
            'showactive': True,
            'bgcolor': '#B0C4DE',  # Cor de fundo do dropdown
            'bordercolor': '#708090',  # Cor da borda
            'font': {'color': '#708090'}  # Cor da fonte
        }],
        title='Clique no Dropdown para selecionar o indicador desejado',
        xaxis_title='Ano',
        yaxis_title='Média dos Indicadores',
        template='plotly_white',
        hovermode='x unified'
    )

    # Todos os traces inicialmente invisíveis
    return fig

# Indicadores a serem analisados
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']

# Chamando a função para plotar os dados com seletor interativo e texto explicativo
plot_interactive_indicator_selector_with_text(means_2020, means_2021, means_2022, means_2023, means_2024, indicators)

# Medias de Indicadores por Idade



In [36]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Lista de indicadores e anos
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']
years = ['2020', '2021', '2022', '2023', '2024']

# Paleta de cores para os indicadores
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880']

# Função para preparar e calcular a média dos dados por idade
def prepare_and_average_data(df, year):
    # Dynamically determine the age column name
    age_col_name = None
    possible_age_cols = [f'IDADE_ALUNO_{year}', 'IDADE_ALUNO']
    for col in possible_age_cols:
        if col in df.columns:
            age_col_name = col
            break

    if not age_col_name:
        # print(f"Warning: No valid age column found for year {year}. Returning empty DataFrame.")
        return pd.DataFrame(), None # Return DataFrame and None for age_col_name

    # Prepare list of actual column names to use for calculations
    actual_indicator_cols = []
    for indicator in indicators:
        suffixed_col = f'{indicator}_{year}'
        unsuffixed_col = indicator
        col_to_process = None

        if suffixed_col in df.columns:
            col_to_process = suffixed_col
        elif unsuffixed_col in df.columns:
            col_to_process = unsuffixed_col
        # If neither is found, we'll skip this indicator for this year.

        if col_to_process:
            actual_indicator_cols.append(col_to_process)

    if not actual_indicator_cols:
        # print(f"Warning: No valid indicator columns found for year {year}. Returning empty DataFrame.")
        return pd.DataFrame(), None # Return DataFrame and None for age_col_name

    # Combine actual indicator columns with the determined age column
    cols_to_use = actual_indicator_cols + [age_col_name]

    # Process data
    data = df[cols_to_use].copy() # Use .copy() to avoid SettingWithCopyWarning

    # Convert indicator columns to numeric, handling commas and errors
    for col in actual_indicator_cols:
        data[col] = pd.to_numeric(data[col].astype(str).str.replace(',', '.'), errors='coerce')

    # Convert age column to numeric, handling errors
    data[age_col_name] = pd.to_numeric(data[age_col_name], errors='coerce')

    data = data.dropna(subset=actual_indicator_cols + [age_col_name]) # Drop NaNs after conversions

    if data.empty:
        # print(f"Warning: No valid numeric data after cleaning for year {year}. Returning empty DataFrame.")
        return pd.DataFrame(), None # Return DataFrame and None for age_col_name

    data[age_col_name] = data[age_col_name].astype(int) # Ensure age is int after conversion and dropping NaNs

    # Rename indicator columns to a consistent format for grouping if they were unsuffixed
    rename_dict = {}
    for original_col in actual_indicator_cols:
        if not original_col.endswith(f'_{year}'): # If it's an unsuffixed column like 'INDE'
            rename_dict[original_col] = f'{original_col}_{year}'
    data = data.rename(columns=rename_dict)

    # Update actual_indicator_cols to their new suffixed names for grouping
    final_indicator_cols = [rename_dict.get(col, col) for col in actual_indicator_cols]

    return data.groupby(age_col_name)[final_indicator_cols].mean().reset_index(), age_col_name

# Função para criar um gráfico com dropdown para selecionar o ano
def plot_data_with_dropdown(df_2020, df_2021, df_2022, df_2023, df_2024):
    fig = go.Figure()

    # Mapeamento dos anos para seus respectivos dataframes
    dataframes = {'2020': df_2020, '2021': df_2021, '2022': df_2022, '2023': df_2023, '2024': df_2024}

    # Criando traços para cada ano e escondendo todos inicialmente, exceto o primeiro ano
    for year_idx, year in enumerate(years):
        df = dataframes[year]
        data, current_age_col_name = prepare_and_average_data(df, year)
        if not data.empty and current_age_col_name is not None:
            for j, indicator in enumerate(indicators):
                # Use the suffixed column name for plotting as prepared by prepare_and_average_data
                column_name_for_plot = f'{indicator}_{year}'
                if column_name_for_plot in data.columns: # Ensure the column exists in the processed data
                    fig.add_trace(
                        go.Scatter(
                            x=data[current_age_col_name].astype(int),  # Use the dynamically determined age column name
                            y=data[column_name_for_plot],
                            mode='lines+markers',
                            name=f'{indicator} ({year})',
                            line=dict(color=colors[j]),
                            visible=(year_idx == 0)  # Apenas o primeiro ano (2020) visível por padrão
                        )
                    )

    # Atualizando layout
    fig.update_layout(
        title=f'Média dos Indicadores de Desempenho por Idade em {years[0]}',
        xaxis=dict(title='Idade', type='linear', tickmode='linear', dtick=1),  # Definir o eixo x como numérico
        yaxis=dict(title='Média'),
        legend_title='Indicador',
        height=450,
        width=1000,
        updatemenus=[  # Adicionando o dropdown
            dict(
                buttons=[
                    dict(label=f"{year}",
                        method="update",
                        args=[{"visible": [k // len(indicators) == year_idx for k in range(len(indicators) * len(years))]},  # Mostrar todos os indicadores do ano correspondente
                            {"title": f'Média dos Indicadores de Desempenho por Idade em {year}'}])
                    for year_idx, year in enumerate(years)
                ],
                direction="down",
                showactive=True,
                x=0.17,
                y=1.15
            )
        ]
    )

    return fig

# Chamar a função para plotar o gráfico com dropdown usando os dataframes
plot_data_with_dropdown(df_2020, df_2021, df_2022, df_2023, df_2024)


# Analise dos Tipos de Pedras

# 1.   Por ano


In [37]:
def load_data(file_path):
    return pd.read_csv(file_path, sep=';', decimal='.', encoding='latin1')

def preprocess_data(df, year):
    pedra_column_suffixed = f'PEDRA_{year}'
    pedra_column_unsuffixed = 'PEDRA'
    actual_pedra_column = None

    if pedra_column_suffixed in df.columns:
        actual_pedra_column = pedra_column_suffixed
    elif pedra_column_unsuffixed in df.columns:
        actual_pedra_column = pedra_column_unsuffixed
    else:
        # If no pedra column found, return an empty series
        print(f"Warning: No 'PEDRA' column found for year {year}. Skipping preprocessing.")
        return pd.Series(dtype=int) # Return an empty series

    df = df[[actual_pedra_column]].dropna()
    df_filtered = df[~df[actual_pedra_column].isin(['D9891/2A', '#NULO!'])]
    return df_filtered[actual_pedra_column].value_counts()

def create_dataframe_from_series(series, year):
    df = pd.DataFrame(series).reset_index()
    df.columns = ['Tipo de Pedra', 'Contagem']
    df['Ano'] = year
    return df

def setup_plot(df_combined, title, colors):
    fig = go.Figure()
    for pedra in df_combined['Tipo de Pedra'].unique():
        df_pedra = df_combined[df_combined['Tipo de Pedra'] == pedra]
        fig.add_trace(go.Bar(
            x=df_pedra['Ano'],
            y=df_pedra['Contagem'],
            name=pedra,
            marker_color=colors.get(pedra, '#000000'),  # Default black if not specified
            text=df_pedra['Contagem'],
            textposition='inside',
            insidetextfont=dict(color='black')  # Set the text color to white
        ))

    fig.update_layout(
        title=title,
        xaxis_title='Ano',
        yaxis_title='Número de Ocorrências',
        barmode='group',
        hovermode='x',
        legend=dict(orientation="h", yanchor="bottom", y=1.00, xanchor="center", x=0.5)

    )
    fig.show()

# Define the colors for each type of stone
cores_pedras = {
    'Quartzo': '#B9F2FF',  # Diamante
    'Ágata': '#FFC0CB',    # Light Pink
    'Ametista': '#9966CC', # Violet
    'Topázio': '#FFD700'   # Gold
}

# Load and preprocess the data
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')

# Apply preprocessing and generate dataframes
pedra_counts_2020 = preprocess_data(df_2020, '2020')
pedra_counts_2021 = preprocess_data(df_2021, '2021')
pedra_counts_2022 = preprocess_data(df_2022, '2022')
pedra_counts_2023 = preprocess_data(df_2023, '2023')
pedra_counts_2024 = preprocess_data(df_2024, '2024')

df_2020 = create_dataframe_from_series(pedra_counts_2020, '2020')
df_2021 = create_dataframe_from_series(pedra_counts_2021, '2021')
df_2022 = create_dataframe_from_series(pedra_counts_2022, '2022')
df_2023 = create_dataframe_from_series(pedra_counts_2023, '2023')
df_2024 = create_dataframe_from_series(pedra_counts_2024, '2024')

# Concatenate dataframes
df_combined = pd.concat([df_2020, df_2021, df_2022, df_2023, df_2024])

# Setup and show the plot
setup_plot(df_combined, 'Comparação de Ocorrências de Pedras por Tipo em Cada Ano', cores_pedras)

# 2.   Genero


In [38]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Reload original dataframes to ensure they have the necessary columns
df_2020 = pd.read_csv('df_2020.csv', sep=';', encoding='latin1')
df_2021 = pd.read_csv('df_2021.csv', sep=';', encoding='latin1')
df_2022 = pd.read_csv('df_2022.csv', sep=';', encoding='latin1')
df_2023 = pd.read_csv('df_2023.csv', sep=';', encoding='latin1')
df_2024 = pd.read_csv('df_2024.csv', sep=';', encoding='latin1')

def get_pedra_col_name(df, year):
    if f'PEDRA_{year}' in df.columns:
        return f'PEDRA_{year}'
    elif 'PEDRA' in df.columns:
        return 'PEDRA'
    return None

def get_gender_col_name(df, year):
    possible_gender_cols = ['GÊNERO', 'Sexo', f'SEXO_ALUNO_{year}']
    for col in possible_gender_cols:
        if col in df.columns:
            return col
    return None

# Define common 'PEDRA' types for filtering
pedra_types = ['Ametista', 'Quartzo', 'Topázio', 'Ágata']

# Create a base DataFrame with all possible Sexo/PEDRA combinations for a complete merge
all_sexo = ['M', 'F']
all_pedra_types = ['Ametista', 'Quartzo', 'Topázio', 'Ágata']
base_comparison = pd.DataFrame([(s, p) for s in all_sexo for p in all_pedra_types], columns=['Sexo', 'PEDRA'])

years_to_process = ['2020', '2021', '2022', '2023', '2024']
dfs_to_process = [df_2020, df_2021, df_2022, df_2023, df_2024]

pedra_comparison = base_comparison.copy() # Start with the base combinations

for year, df_year in zip(years_to_process, dfs_to_process):
    pedra_col = get_pedra_col_name(df_year, year)
    gender_col = get_gender_col_name(df_year, year)

    if pedra_col and gender_col:
        df_clean = df_year.copy()

        # Standardize gender values
        df_clean['Sexo_Clean'] = df_clean[gender_col].astype(str).str.upper().str.strip()
        df_clean['Sexo_Clean'] = df_clean['Sexo_Clean'].map({
            'MASCULINO': 'M',
            'M': 'M',
            'FEMININO': 'F',
            'F': 'F'
        }).fillna(np.nan)

        # Filter for valid gender and pedra types
        df_clean = df_clean.dropna(subset=['Sexo_Clean'])
        df_clean = df_clean[df_clean[pedra_col].isin(pedra_types)]

        if not df_clean.empty:
            grouped_counts = df_clean.groupby(['Sexo_Clean', pedra_col]).size().reset_index(name=f'count_{year}')
            grouped_counts.rename(columns={'Sexo_Clean': 'Sexo', pedra_col: 'PEDRA'}, inplace=True)

            # Merge current year's counts into the main comparison DataFrame
            pedra_comparison = pd.merge(pedra_comparison, grouped_counts, on=['Sexo', 'PEDRA'], how='left')
        else:
            # If df_clean is empty, ensure the count column for this year is added with NaNs
            pedra_comparison[f'count_{year}'] = np.nan
    else:
        # If no pedra_col or gender_col found, add the column with NaNs
        pedra_comparison[f'count_{year}'] = np.nan

# After the loop, fill any remaining NaNs from left merges or missing years with 0
count_cols = [f'count_{y}' for y in years_to_process]
pedra_comparison[count_cols] = pedra_comparison[count_cols].fillna(0)

# Calculate total for all years for consistency
pedra_comparison['total'] = pedra_comparison[count_cols].sum(axis=1)

# Create a dataframe separated by gender, with the total of each stone
totais_feminino = pedra_comparison[pedra_comparison['Sexo'] == 'F'].groupby('PEDRA')['total'].sum().reset_index()
totais_masculino = pedra_comparison[pedra_comparison['Sexo'] == 'M'].groupby('PEDRA')['total'].sum().reset_index()

# Criando a figura para barras empilhadas por pedra e por gênero (sem separar por ano)
fig = go.Figure()

# Adicionando dados do total para o gênero feminino
fig.add_trace(go.Bar(
    x=totais_feminino['PEDRA'],
    y=totais_feminino['total'],
    name='Feminino',
    marker_color='#8B008B',
    hovertemplate='<b>Pedra:</b> %{x}<br><b>Sexo:</b> Feminino<br><b>Quantidade:</b> %{y}<extra></extra>'
))

# Adicionando dados do total para o gênero masculino
fig.add_trace(go.Bar(
    x=totais_masculino['PEDRA'],
    y=totais_masculino['total'],
    name='Masculino',
    marker_color='#0000CD',
    hovertemplate='<b>Pedra:</b> %{x}<br><b>Sexo:</b> Masculino<br><b>Quantidade:</b> %{y}<extra></extra>'
))

# Atualizando o layout para barras empilhadas
fig.update_layout(
    barmode='stack',
    title='Distribuição Total de Estudantes por Pedra e Gênero (2020-2024)',
    xaxis_title='Tipo de Pedra',
    yaxis_title='Número de Estudantes',
    template='plotly_white',
    legend_title='Gênero',
    #height=700
)
fig.show()

# 3.   Idade

In [39]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px # Import plotly.express

# Função para carregar os dados
def load_data(filepath):
    return pd.read_csv(filepath, sep=';', encoding='latin1')

# Função para converter as idades para numérico, ignorando erros
def limpar_idade(df, coluna_idade):
    df[coluna_idade] = pd.to_numeric(df[coluna_idade], errors='coerce')  # Converte para float e ignora erros
    df = df.dropna(subset=[coluna_idade])  # Remove linhas onde a idade é NaN
    return df

# Cores personalizadas para cada tipo de pedra
cores_pedras = {
    'Quartzo': '#B9F2FF',  # Diamante
    'Ágata': '#FFC0CB',    # Light Pink
    'Ametista': '#9966CC', # Violet
    'Topázio': '#FFD700'   # Gold
}

# Carregar os dados
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')

# Helper function to get the correct column name (suffixed or unsuffixed)
def get_dynamic_col_name(df, base_name, year):
    suffixed_name = f'{base_name}_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif base_name in df.columns:
        return base_name
    return None

# List of dataframes and years
dfs = {
    '2020': df_2020,
    '2021': df_2021,
    '2022': df_2022,
    '2023': df_2023,
    '2024': df_2024
}
years = ['2020', '2021', '2022', '2023', '2024']

all_dfs_grouped = []

for year in years:
    df_year = dfs[year]
    idade_col = get_dynamic_col_name(df_year, 'IDADE_ALUNO', year)
    pedra_col = get_dynamic_col_name(df_year, 'PEDRA', year)

    # Dynamically determine the 'NOME' column name
    name_col = None
    if 'NOME' in df_year.columns:
        name_col = 'NOME'
    elif 'Nome Anonimizado' in df_year.columns:
        name_col = 'Nome Anonimizado'

    if idade_col and pedra_col and name_col:
        df_clean = df_year[[name_col, idade_col, pedra_col]].copy()
        df_clean.rename(columns={name_col: 'NOME', idade_col: 'IDADE', pedra_col: 'PEDRA'}, inplace=True)

        df_clean = limpar_idade(df_clean, 'IDADE')
        if not df_clean.empty:
            df_grouped = df_clean.groupby(['IDADE', 'PEDRA']).size().reset_index(name=f'count_{year}')
            all_dfs_grouped.append(df_grouped)
    else:
        print(f"Warning: Missing 'NOME', 'IDADE_ALUNO' or 'PEDRA' column for year {year}. Skipping.")
        # If any essential column is missing, create an empty grouped dataframe for this year
        # This is important for the merge to ensure all years are accounted for in df_combined
        empty_grouped = pd.DataFrame(columns=['IDADE', 'PEDRA', f'count_{year}'])
        all_dfs_grouped.append(empty_grouped)

# Fazer merge das contagens por idade e pedra
if all_dfs_grouped:
    df_combined = all_dfs_grouped[0]
    for i in range(1, len(all_dfs_grouped)):
        df_combined = pd.merge(df_combined, all_dfs_grouped[i], on=['IDADE', 'PEDRA'], how='outer')
else:
    df_combined = pd.DataFrame(columns=['IDADE', 'PEDRA'] + [f'count_{y}' for y in years])

# Preencher valores nulos com 0
df_combined = df_combined.fillna(0)

# Somar as contagens para os 5 anos
df_combined['total_count'] = df_combined[[f'count_{y}' for y in years]].sum(axis=1)

# Gerar o histograma utilizando Plotly com bins de 1 ano (intervalo de idade)
fig = px.histogram(df_combined,
                x='IDADE',
                y='total_count',
                color='PEDRA',
                color_discrete_map=cores_pedras,
                barmode='stack',
                title="Distribuição de Idades por Pedra (2020-2024)", # Update title to reflect all years
                nbins=len(df_combined['IDADE'].unique()),  # Definir bins de 1 em 1
                )

# Atualizar títulos e rótulos
fig.update_layout(
    xaxis_title="Idade",
    yaxis_title="Quantidade Total de Alunos",
    template='plotly_white',
    xaxis=dict(
                tickmode='linear',  # Configura os ticks como lineares
                dtick=1  # Mostra ticks de 1 em 1
            )
)
fig.show()


# 4.  Transicao de Pedras

In [40]:
# Função para carregar os dados
def load_data(filepath):
    return pd.read_csv(filepath, sep=';', encoding='latin1')

# Função para converter as idades para numérico, ignorando erros
def limpar_idade(df, coluna_idade):
    df[coluna_idade] = pd.to_numeric(df[coluna_idade], errors='coerce')  # Converte para float e ignora erros
    df = df.dropna(subset=[coluna_idade])  # Remove linhas onde a idade é NaN
    return df

# Cores personalizadas para cada tipo de pedra
cores_pedras = {
    'Quartzo': '#B9F2FF',  # Diamante
    'Ágata': '#FFC0CB',    # Light Pink
    'Ametista': '#9966CC', # Violet
    'Topázio': '#FFD700'   # Gold
}

# Carregar os dados
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')

# Helper function to get the correct 'PEDRA' column name (suffixed or unsuffixed)
def get_pedra_col_name(df, year):
    suffixed_name = f'PEDRA_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif 'PEDRA' in df.columns:
        return 'PEDRA'
    return None

# Helper function to get the correct 'NOME' column name
def get_name_col_name(df):
    if 'NOME' in df.columns:
        return 'NOME'
    elif 'Nome Anonimizado' in df.columns:
        return 'Nome Anonimizado'
    return None

# Mapeamento para ordem das pedras
ordem_pedras = {'Quartzo': 1, 'Ágata': 2, 'Ametista': 3, 'Topázio': 4}

# Process each year's dataframe to get clean 'NOME', 'PEDRA', and 'PEDRA_NUM'
dfs_years_raw = {
    '2020': df_2020,
    '2021': df_2021,
    '2022': df_2022,
    '2023': df_2023,
    '2024': df_2024
}

processed_dfs = {}
for year_str, df_year in dfs_years_raw.items():
    name_col = get_name_col_name(df_year)
    pedra_col = get_pedra_col_name(df_year, year_str)

    if name_col and pedra_col:
        df_clean_year = df_year[[name_col, pedra_col]].copy()
        df_clean_year.rename(columns={name_col: 'NOME', pedra_col: 'PEDRA'}, inplace=True)
        df_clean_year['NOME'] = df_clean_year['NOME'].str.upper() # Standardize NOME to uppercase
        df_clean_year['PEDRA_NUM'] = df_clean_year['PEDRA'].map(ordem_pedras)
        processed_dfs[year_str] = df_clean_year
    else:
        print(f"Warning: Missing 'NOME' or 'PEDRA' column for year {year_str}. Skipping processing for this year.")
        processed_dfs[year_str] = pd.DataFrame(columns=['NOME', 'PEDRA', 'PEDRA_NUM']) # Ensure empty DF has expected columns

# Mesclar os dataframes para comparar a pedra de cada aluno entre os anos
# Check if data exists for necessary years before merging

df_2020_2021 = pd.DataFrame()
if '2020' in processed_dfs and '2021' in processed_dfs:
    df_2020_2021 = pd.merge(processed_dfs['2020'], processed_dfs['2021'][['NOME', 'PEDRA_NUM']], on='NOME', how='inner', suffixes=('_2020', '_2021'))

df_2021_2022 = pd.DataFrame()
if '2021' in processed_dfs and '2022' in processed_dfs:
    df_2021_2022 = pd.merge(processed_dfs['2021'], processed_dfs['2022'][['NOME', 'PEDRA_NUM']], on='NOME', how='inner', suffixes=('_2021', '_2022'))

df_2022_2023 = pd.DataFrame()
if '2022' in processed_dfs and '2023' in processed_dfs:
    df_2022_2023 = pd.merge(processed_dfs['2022'], processed_dfs['2023'][['NOME', 'PEDRA_NUM']], on='NOME', how='inner', suffixes=('_2022', '_2023'))

df_2023_2024 = pd.DataFrame()
if '2023' in processed_dfs and '2024' in processed_dfs:
    df_2023_2024 = pd.merge(processed_dfs['2023'], processed_dfs['2024'][['NOME', 'PEDRA_NUM']], on='NOME', how='inner', suffixes=('_2023', '_2024'))


# Função para classificar as mudanças
def classificar_movimento(row, col1, col2):
    # Handle NaN values that might arise from missing data in merges
    if pd.isna(row[col1]) or pd.isna(row[col2]):
        return 'Sem dados'
    elif row[col1] < row[col2]:
        return 'Subiu'
    elif row[col1] > row[col2]:
        return 'Desceu'
    else:
        return 'Sem mudança'

# Aplicar a classificação, ensuring dataframes are not empty
movimento_2020_2021 = df_2020_2021.apply(classificar_movimento, col1='PEDRA_NUM_2020', col2='PEDRA_NUM_2021', axis=1).value_counts() if not df_2020_2021.empty else pd.Series(dtype=int)
movimento_2021_2022 = df_2021_2022.apply(classificar_movimento, col1='PEDRA_NUM_2021', col2='PEDRA_NUM_2022', axis=1).value_counts() if not df_2021_2022.empty else pd.Series(dtype=int)
movimento_2022_2023 = df_2022_2023.apply(classificar_movimento, col1='PEDRA_NUM_2022', col2='PEDRA_NUM_2023', axis=1).value_counts() if not df_2022_2023.empty else pd.Series(dtype=int)
movimento_2023_2024 = df_2023_2024.apply(classificar_movimento, col1='PEDRA_NUM_2023', col2='PEDRA_NUM_2024', axis=1).value_counts() if not df_2023_2024.empty else pd.Series(dtype=int)


# Preparar os dados para o gráfico
movimentos_data = pd.DataFrame({
    'Movimento': ['Subiu', 'Desceu', 'Sem mudança'],
    '2020-2021': [movimento_2020_2021.get('Subiu', 0), movimento_2020_2021.get('Desceu', 0), movimento_2020_2021.get('Sem mudança', 0)],
    '2021-2022': [movimento_2021_2022.get('Subiu', 0), movimento_2021_2022.get('Desceu', 0), movimento_2021_2022.get('Sem mudança', 0)],
    '2022-2023': [movimento_2022_2023.get('Subiu', 0), movimento_2022_2023.get('Desceu', 0), movimento_2022_2023.get('Sem mudança', 0)],
    '2023-2024': [movimento_2023_2024.get('Subiu', 0), movimento_2023_2024.get('Desceu', 0), movimento_2023_2024.get('Sem mudança', 0)]


})

display(movimentos_data)

# Definir as cores do semáforo: Verde (Subiu), Vermelho (Desceu), Amarelo (Sem mudança)
cores_semaforo = {'Subiu': '#228B22', 'Desceu': '#B22222', 'Sem mudança': '#FFD700'}

# Plotar o gráfico de barras com as cores do semáforo
fig = go.Figure()

for movimento in movimentos_data['Movimento']:
    fig.add_trace(go.Bar(
        x=['2020-2021', '2021-2022', '2022-2023', '2023-2024'],
        y=movimentos_data.loc[movimentos_data['Movimento'] == movimento].iloc[0, 1:].values,
        name=movimento,
        marker_color=cores_semaforo[movimento]
    ))

# Atualizar layout do gráfico
fig.update_layout(
    title_text='Movimentos de Alunos Entre as Pedras (2020-2024)', # Updated title to reflect all years
    #title_x=0.5,
    xaxis_title='Ano',
    yaxis_title='Quantidade de Alunos',
    #barmode='stack',
    template='plotly_white'
)
fig.show()

,Movimento,2020-2021,2021-2022,2022-2023,2023-2024
0,Subiu,72,165,139,164
1,Desceu,152,144,139,175
2,Sem mudança,231,123,292,339


# Analise das Evasoes Escolares

# 1.   Motivos


In [41]:
# Carregar os dados
def load_data(filepath, sep=';', encoding='latin1'):
    return pd.read_csv(filepath, sep=sep, encoding=encoding)
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')
tbmotivoinativacao = load_data('TbMotivoInativacao.csv', sep=',')
tbalunoturma = load_data('TbAlunoTurma.csv', sep=',')
tbaluno = load_data('TbAluno.csv', sep=',')

# 1. Fazer o merge entre TbMotivoInativacao e TbAlunoTurma pelo IdMotivoInativacao
tb_aluno_turma = pd.merge(tbalunoturma, tbmotivoinativacao[['IdMotivoInativacao', 'MotivoInativacao']],
                        left_on='IdMotivoInativacao', right_on='IdMotivoInativacao', how='left')


# 2. Tratar a coluna NomeAluno em TbAluno (ex: 'Aluno 1' para 'Aluno-1' e deixar em upper case)
tbaluno['NomeAluno'] = tbaluno['NomeAluno'].apply(lambda x: x.replace(' ', '-').upper())

# 3. Levar a coluna MotivoInativacao de TbAlunoTurma para TbAluno pela coluna IdAluno
tb_aluno = pd.merge(tbaluno, tb_aluno_turma[['IdAluno', 'MotivoInativacao']],
                    left_on='IdAluno', right_on='IdAluno', how='left')


# Contagem dos motivos de inativação na tabela atualizada
motivos_inativacao_count = tb_aluno['MotivoInativacao'].value_counts().reset_index(name='Total')

# Renomear colunas para exibição no gráfico
motivos_inativacao_count.columns = ['Motivo', 'Total']

# Criando gráfico de barras horizontal com cores nas barras
fig = px.bar(motivos_inativacao_count, x='Total', y='Motivo', orientation='h',
            title='Contagem dos Motivos de Inativação',
            labels={'Total': 'Total de Inativos'},
            color='Motivo',  # Definir cores com base nos motivos
            color_discrete_sequence=px.colors.qualitative.Plotly)  # Usar a paleta de cores padrão do Plotly
# Ajustar a margem esquerda para que os nomes dos motivos apareçam corretamente
fig.update_layout(margin=dict(l=350), showlegend=False)
# Exibir o gráfico
fig.show()

FileNotFoundError: [Errno 2] No such file or directory: 'TbMotivoInativacao.csv'

# 2.  Idade

In [ ]:
# Carregar os dados
def load_data(filepath, sep=';', encoding='latin1'):
    return pd.read_csv(filepath, sep=sep, encoding=encoding)
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')
tbmotivoinativacao = load_data('TbMotivoInativacao.csv', sep=',')
# Correct garbled characters in MotivoInativacao column if needed
tbmotivoinativacao['MotivoInativacao'] = tbmotivoinativacao['MotivoInativacao'].apply(lambda x: x.encode('latin1').decode('utf-8') if isinstance(x, str) else x)
tbalunoturma = load_data('TbAlunoTurma.csv', sep=',')
tbaluno = load_data('TbAluno.csv', sep=',')

# Helper function to get the correct age column name (suffixed or unsuffixed)
def get_age_col_name(df, year):
    suffixed_name = f'IDADE_ALUNO_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif 'IDADE_ALUNO' in df.columns:
        return 'IDADE_ALUNO'
    return None

# Helper function to get the correct student ID column name
def get_id_col_name(df, year):
    suffixed_name = f'ID_ALUNO_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif 'IdAluno' in df.columns: # Assuming some df_year might directly have 'IdAluno'
        return 'IdAluno'
    return None

# Helper function to get the correct student name column (NOME or Nome Anonimizado)
def get_name_col_name(df):
    if 'NOME' in df.columns:
        return 'NOME'
    elif 'Nome Anonimizado' in df.columns:
        return 'Nome Anonimizado'
    return None

# Passo 1: Merge entre `TbAluno` e `tb_aluno_turma` para trazer o IdMotivoInativacao

# Standardize NomeAluno in tbaluno before any merges (important for name-based merges if needed)
tbaluno['NomeAluno'] = tbaluno['NomeAluno'].astype(str).str.replace(' ', '-').str.upper()

tb_aluno_merged = pd.merge(tbaluno, tbalunoturma[['IdAluno', 'IdMotivoInativacao']], on='IdAluno', how='left')

# Passo 2: Merge com `tb_motivo_inativacao` para trazer os motivos de inativação
tb_aluno_merged = pd.merge(tb_aluno_merged, tbmotivoinativacao[['IdMotivoInativacao', 'MotivoInativacao']], on='IdMotivoInativacao', how='left')

# Passo 3: Trazer as idades dos dataframes, tratando nomes de colunas dinamicamente
dfs_years_raw = {
    '2020': df_2020,
    '2021': df_2021,
    '2022': df_2022,
    '2023': df_2023,
    '2024': df_2024
}
alle_idades_dfs = []

for year, df_year in dfs_years_raw.items():
    age_col = get_age_col_name(df_year, year)
    id_col = get_id_col_name(df_year, year)
    name_col = get_name_col_name(df_year)

    temp_df = pd.DataFrame() # Initialize empty temp_df

    if age_col:
        if id_col: # Prefer merging by IdAluno directly if available in df_year
            temp_df = df_year[[id_col, age_col]].copy()
            temp_df.rename(columns={id_col: 'IdAluno', age_col: 'Idade'}, inplace=True)
            # Ensure IdAluno is numeric for consistent merging
            temp_df['IdAluno'] = pd.to_numeric(temp_df['IdAluno'], errors='coerce')
            temp_df.dropna(subset=['IdAluno'], inplace=True) # Drop rows where IdAluno could not be converted

        elif name_col: # Fallback to name-based merge if no direct IdAluno in df_year
            temp_df = df_year[[name_col, age_col]].copy()
            temp_df.rename(columns={name_col: 'NomeAluno', age_col: 'Idade'}, inplace=True)
            temp_df['NomeAluno'] = temp_df['NomeAluno'].astype(str).str.replace(' ', '-').str.upper()

            # Merge with tbaluno to get IdAluno
            temp_df = pd.merge(temp_df, tbaluno[['IdAluno', 'NomeAluno']], on='NomeAluno', how='inner')

        if not temp_df.empty: # Only append if temp_df has data after processing
            temp_df = temp_df.drop_duplicates(subset=['IdAluno', 'Idade'])
            alle_idades_dfs.append(temp_df[['IdAluno', 'Idade']])
        else:
            print(f"Warning: No valid student data (age or ID) found for year {year} after processing. Skipping.")
    else:
        print(f"Warning: Missing age column for year {year}. Skipping age data for this year.")


# Concatenar as idades dos diferentes anos em um único dataframe
df_idades = pd.DataFrame(columns=['IdAluno', 'Idade'])
if alle_idades_dfs:
    df_idades = pd.concat(alle_idades_dfs, ignore_index=True)
    # For each IdAluno, take the maximum age to ensure a single age per student
    df_idades = df_idades.groupby('IdAluno')['Idade'].max().reset_index()
    df_idades['IdAluno'] = df_idades['IdAluno'].astype(int) # Ensure IdAluno is int

# Converter a coluna 'Idade' para numérico e tratar valores inválidos
df_idades['Idade'] = pd.to_numeric(df_idades['Idade'], errors='coerce')

# Fazer merge do dataframe com as idades no dataframe `tb_aluno_merged`
tb_aluno_final = pd.merge(tb_aluno_merged, df_idades, on='IdAluno', how='left')

# Passo 4: Criar a faixa etária e contar os motivos de inativação
age_bins = [0, 11, 16, 21] # Adjusted upper bound to include age 20 in the last bin, aligned with labels
age_labels = ['0-10', '11-15', '16-20']
# Filter out NaNs from Idade and MotivoInativacao before creating Faixa_Etaria
tb_aluno_final_filtered = tb_aluno_final.dropna(subset=['Idade', 'MotivoInativacao']).copy()

tb_aluno_final_filtered['Faixa_Etaria'] = pd.cut(tb_aluno_final_filtered['Idade'], bins=age_bins, labels=age_labels, right=False, include_lowest=True) # Use right=False to include 0 in '0-10' and include_lowest=True

motivos_por_idade = tb_aluno_final_filtered.groupby(['Faixa_Etaria', 'MotivoInativacao']).size().reset_index(name='Total')

# Get all unique motives from the original TbMotivoInativacao for completeness
all_motives = tbmotivoinativacao['MotivoInativacao'].unique()

# Create a complete DataFrame with all combinations of Faixa_Etaria and MotivoInativacao
all_combinations = pd.MultiIndex.from_product([age_labels, all_motives], names=['Faixa_Etaria', 'MotivoInativacao']).to_frame(index=False)

# Merge with the calculated motives_por_idade and fill NaN with 0
motivos_por_idade = pd.merge(all_combinations, motivos_por_idade, on=['Faixa_Etaria', 'MotivoInativacao'], how='left').fillna(0)

display(motivos_por_idade)

# Identificar os motivos únicos
motivos_unicos = motivos_por_idade['MotivoInativacao'].unique()

# Criar um mapa de cores dinâmico (usando a paleta Plotly)
cores = px.colors.qualitative.Plotly

# Mapear cores para todos os motivos de inativação (usando ciclo se houver mais motivos do que cores)
color_map = {motivo: cores[i % len(cores)] for i, motivo in enumerate(motivos_unicos)}

# Passo 5: Gerar gráfico de barras com faixas etárias e motivos de inativação
fig = px.bar(motivos_por_idade, x='Faixa_Etaria', y='Total', color='MotivoInativacao', barmode='group',
             title='Faixa Etária e Motivos de Inativação na ONG', labels={'Total': 'Quantidade de Evasões', 'Faixa_Etaria': 'Faixa Etária'},
             color_discrete_map=color_map, text_auto=True) # Use text_auto=True for automatic text display

# No need for fig.update_traces(textposition='outside') with text_auto=True

# Ensure categorical order for Faixa_Etaria on the x-axis
fig.update_xaxes(categoryorder='array', categoryarray=age_labels)

# Update layout for increased height and width to provide more space for text labels
fig.update_layout(height=800, width=1400) # Increased width further

# Exibir o gráfico
fig.show()

# Quantidade de Alunos que Melhoraram ou Pioraram nos Indices por Ano

# 2020-2021


In [ ]:
# Função para carregar os dados
def load_data(filepath):
    return pd.read_csv(filepath, sep=';', encoding='latin1')

# Carregar os dataframes para todos os anos
dfs_raw = {
    '2020': load_data('df_2020.csv'),
    '2021': load_data('df_2021.csv'),
    '2022': load_data('df_2022.csv'),
    '2023': load_data('df_2023.csv'),
    '2024': load_data('df_2024.csv')
}

# Standardize 'NOME' column to uppercase in all raw dataframes
for year, df in dfs_raw.items():
    name_col = None
    if 'NOME' in df.columns:
        name_col = 'NOME'
    elif 'Nome Anonimizado' in df.columns:
        name_col = 'Nome Anonimizado'

    if name_col:
        df[name_col] = df[name_col].astype(str).str.upper()
        # Rename to 'NOME' for consistency if it was 'Nome Anonimizado'
        if name_col != 'NOME':
            df.rename(columns={name_col: 'NOME'}, inplace=True)

# Lista de indicadores
indicators = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']

# Helper function to get the correct indicator column name (suffixed or unsuffixed)
def get_indicator_col_name(df, indicator, year):
    suffixed_name = f'{indicator}_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif indicator in df.columns:
        return indicator
    return None

# Helper function to get the correct student name column
def get_name_col_name(df):
    if 'NOME' in df.columns:
        return 'NOME'
    elif 'Nome Anonimizado' in df.columns:
        return 'Nome Anonimizado'
    return None

# Loop through each year pair (e.g., 2020-2021, 2021-2022, etc.)
years = ['2020', '2021', '2022', '2023', '2024']
for i in range(len(years) - 1):
    year1 = years[i]
    year2 = years[i+1]
    df1 = dfs_raw[year1].copy()
    df2 = dfs_raw[year2].copy()

    print(f"Processing comparison for years: {year1}-{year2}")

    # The 'NOME' column should already be standardized by the initial loop
    # No need to dynamically get name_col1, name_col2 or rename again here

    # Collect existing indicator columns for year1 and year2
    cols_to_merge1 = ['NOME']
    cols_to_merge2 = ['NOME']
    actual_indicators1 = {} # Maps original indicator name to actual column name in df1
    actual_indicators2 = {} # Maps original indicator name to actual column name in df2

    for indicator in indicators:
        col1 = get_indicator_col_name(df1, indicator, year1)
        col2 = get_indicator_col_name(df2, indicator, year2)

        if col1:
            cols_to_merge1.append(col1)
            actual_indicators1[indicator] = col1
        else:
            print(f"Warning: Indicator '{indicator}' not found in {year1} data. Skipping for this year pair.")

        if col2:
            cols_to_merge2.append(col2)
            actual_indicators2[indicator] = col2
        else:
            print(f"Warning: Indicator '{indicator}' not found in {year2} data. Skipping for this year pair.")

    # Filter dataframes for relevant columns
    df1_filtered = df1[cols_to_merge1]
    df2_filtered = df2[cols_to_merge2]

    # Rename indicator columns in df1_filtered and df2_filtered for consistent merging
    rename_map1 = {v: f'{k}_{year1}' for k, v in actual_indicators1.items()}
    rename_map2 = {v: f'{k}_{year2}' for k, v in actual_indicators2.items()}

    df1_filtered.rename(columns=rename_map1, inplace=True)
    df2_filtered.rename(columns=rename_map2, inplace=True)

    # Merge the dataframes for the current year pair
    df_merged = pd.merge(df1_filtered, df2_filtered, on='NOME', how='inner')

    if df_merged.empty:
        print(f"No common students found for {year1}-{year2} comparison. Skipping plot.")
        continue

    # Convert the relevant indicator columns to numeric for calculation
    for indicator in indicators:
        col_year1 = f'{indicator}_{year1}'
        col_year2 = f'{indicator}_{year2}'

        if col_year1 in df_merged.columns:
            df_merged[col_year1] = pd.to_numeric(df_merged[col_year1].astype(str).str.replace(',', '.'), errors='coerce')
        if col_year2 in df_merged.columns:
            df_merged[col_year2] = pd.to_numeric(df_merged[col_year2].astype(str).str.replace(',', '.'), errors='coerce')

    # Calculate differences and count improved/worsened students
    melhoraram = {}
    pioraram = {}

    current_comparison_indicators = [ind for ind in indicators if f'{ind}_{year1}' in df_merged.columns and f'{ind}_{year2}' in df_merged.columns]

    for indicator in current_comparison_indicators:
        col_year1 = f'{indicator}_{year1}'
        col_year2 = f'{indicator}_{year2}'

        # Calculate difference only if both columns exist
        df_merged[f'diff_{indicator}'] = df_merged[col_year2] - df_merged[col_year1]

        # Count improvements and deteriorations, dropping NaNs to only compare valid pairs
        valid_diffs = df_merged[f'diff_{indicator}'].dropna()
        melhoraram[indicator] = (valid_diffs > 0).sum()
        pioraram[indicator] = (valid_diffs < 0).sum()

    if not melhoraram and not pioraram:
        print(f"No valid indicator data for comparison in {year1}-{year2}. Skipping plot.")
        continue

    # Create the bar plot for the current year pair
    traces = [
        go.Bar(
            x=list(melhoraram.keys()),
            y=list(melhoraram.values()),
            name='Melhoraram',
            marker_color='green'
        ),
        go.Bar(
            x=list(pioraram.keys()),
            y=list(pioraram.values()),
            name='Pioraram',
            marker_color='red'
        )
    ]

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f'Quantidade de Alunos que Melhoraram ou Pioraram nos Índices ({year1}-{year2})',
        xaxis_title='Indicador',
        yaxis_title='Quantidade de Alunos',
        barmode='group'
    )
    fig.show()

# Top 20 Melhores e piores alunos pelo INDE


In [ ]:
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')

# Helper function to get the correct indicator column name (suffixed or unsuffixed)
def get_indicator_col_name(df, indicator, year):
    suffixed_name = f'{indicator}_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif indicator in df.columns:
        return indicator
    return None

# Helper function to get the correct student name column
def get_name_col_name(df):
    if 'NOME' in df.columns:
        return 'NOME'
    elif 'Nome Anonimizado' in df.columns:
        return 'Nome Anonimizado'
    return None

# Função para garantir que as colunas são numéricas e tratar o separador decimal
def ensure_numeric(df, columns):
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column].astype(str).str.replace(',', '.'), errors='coerce')
    return df

# Função para calcular a variação entre os anos e encontrar as maiores subidas e descidas
def get_top_changes_indice_inde(df_year1, df_year2, year1_str, year2_str):
    # Dynamically get INDE column names
    inde_col_year1 = get_indicator_col_name(df_year1, 'INDE', year1_str)
    inde_col_year2 = get_indicator_col_name(df_year2, 'INDE', year2_str)

    # Dynamically get name column names
    name_col_year1 = get_name_col_name(df_year1)
    name_col_year2 = get_name_col_name(df_year2)

    if not inde_col_year1 or not inde_col_year2 or not name_col_year1 or not name_col_year2:
        print(f"Warning: Could not find required columns for {year1_str}-{year2_str}. Skipping.")
        return pd.DataFrame() # Return empty DataFrame if columns are missing

    # Filter and rename columns for year1
    df_year1_filtered = df_year1[[name_col_year1, inde_col_year1]].copy()
    df_year1_filtered.rename(columns={name_col_year1: 'NOME', inde_col_year1: year1_str}, inplace=True)
    df_year1_filtered['NOME'] = df_year1_filtered['NOME'].astype(str).str.upper() # Standardize NOME

    # Filter and rename columns for year2
    df_year2_filtered = df_year2[[name_col_year2, inde_col_year2]].copy()
    df_year2_filtered.rename(columns={name_col_year2: 'NOME', inde_col_year2: year2_str}, inplace=True)
    df_year2_filtered['NOME'] = df_year2_filtered['NOME'].astype(str).str.upper() # Standardize NOME

    # Juntar os dados em um único dataframe baseado no nome do aluno
    df_combined = pd.merge(df_year1_filtered, df_year2_filtered, on='NOME', how='inner')

    print(f"  Shape after inner merge for {year1_str}-{year2_str}: {df_combined.shape}")
    if not df_combined.empty:
        print(f"  Head after inner merge:\n{df_combined.head()}")

    # Convert the columns to numeric before checking for NaNs
    df_combined = ensure_numeric(df_combined, [year1_str, year2_str])

    if not df_combined.empty:
        print(f"  NaNs in '{year1_str}' column before dropna: {df_combined[year1_str].isnull().sum()}")
        print(f"  NaNs in '{year2_str}' column before dropna: {df_combined[year2_str].isnull().sum()}")

    # Remover valores ausentes
    df_combined = df_combined.dropna()

    print(f"  Shape after dropna for {year1_str}-{year2_str}: {df_combined.shape}")

    if df_combined.empty:
        print(f"No common students with valid INDE data found for {year1_str}-{year2_str}. Skipping.")
        return pd.DataFrame()

    # Calcular a variação
    df_combined['Variação'] = df_combined[year2_str] - df_combined[year1_str]

    # Selecionar os top 20 maiores subidas e top 20 maiores descidas
    top_20_subidas = df_combined.nlargest(20, 'Variação')[['NOME', 'Variação']].assign(Tipo='Subida')
    top_20_descidas = df_combined.nsmallest(20, 'Variação')[['NOME', 'Variação']].assign(Tipo='Descida')

    # Combinar as subidas e descidas
    top_changes = pd.concat([top_20_subidas, top_20_descidas])

    return top_changes

# Define a list of years from 2020 to 2024
years = ['2020', '2021', '2022', '2023', '2024']

# Create a dictionary to store the dataframes for each year
dfs_raw = {
    '2020': df_2020,
    '2021': df_2021,
    '2022': df_2022,
    '2023': df_2023,
    '2024': df_2024
}

# Iterate through each year pair from 2020-2021 to 2023-2024
for i in range(len(years) - 1):
    year1 = years[i]
    year2 = years[i+1]

    print(f"\nProcessing comparison for years: {year1}-{year2}")

    # Get the dataframes for the current year pair
    df_year1 = dfs_raw[year1].copy()
    df_year2 = dfs_raw[year2].copy()

    # Calculate the top 20 changes for the current year pair
    top_changes = get_top_changes_indice_inde(df_year1, df_year2, year1, year2)

    # Check if the returned top_changes DataFrame is empty
    if top_changes.empty:
        print(f"No sufficient data to generate the plot for {year1}-{year2}.")
        continue

    # Generate and display the bar chart
    fig = px.bar(top_changes, x='Variação', y='NOME', color='Tipo', orientation='h',
                color_discrete_map={'Subida': 'green', 'Descida': 'red'},
                title=f'Top 20 Maiores Subidas e Descidas do INDE ({year1}-{year2})',
                labels={'Variação': 'Variação do INDE'})

    # Update layout for better visualization
    fig.update_layout(bargap=0.2, margin=dict(l=100))
    fig.show()

# Ponto de Virada

In [ ]:
import pandas as pd
import plotly.graph_objs as go

# Função para carregar os dados
def load_data(filepath):
    return pd.read_csv(filepath, sep=';', encoding='latin1')

# Helper function to get the correct column name (suffixed or unsuffixed)
def get_column_name(df, base_name, year):
    suffixed_name = f'{base_name}_{year}'
    if suffixed_name in df.columns:
        return suffixed_name
    elif base_name in df.columns:
        return base_name
    return None

# Carregar os dados
df_2020 = load_data('df_2020.csv')
df_2021 = load_data('df_2021.csv')
df_2022 = load_data('df_2022.csv')
df_2023 = load_data('df_2023.csv')
df_2024 = load_data('df_2024.csv')


# Dynamically determine the correct column names for 'BOLSISTA' and 'PONTO_VIRADA' in df_2022
bolsista_col_name = get_column_name(df_2022, 'Indicado', '2022') # Use 'Indicado' as the base name for bolsista_col_name
ponto_virada_col_name = get_column_name(df_2022, 'ATINGIU_PV', '2022') # Use 'ATINGIU_PV' as the base name

if bolsista_col_name is None:
    print(f"Error: 'BOLSISTA' (or 'Indicado') column not found in df_2022. Please check the dataframe structure. Available columns: {df_2022.columns.tolist()}")
elif ponto_virada_col_name is None:
    print(f"Error: 'PONTO_VIRADA' (or 'ATINGIU_PV') column not found in df_2022. Please check the dataframe structure. Available columns: {df_2022.columns.tolist()}")
else:
    # Contagem de alunos que atingiram ou não o Ponto de Virada por categoria (bolsistas ou não bolsistas)
    # Ensure to drop NaNs in the relevant columns before grouping
    df_filtered = df_2022.dropna(subset=[bolsista_col_name, ponto_virada_col_name]).copy()

    # Normalize 'Indicado' column to 'Sim'/'Não'
    df_filtered[bolsista_col_name] = df_filtered[bolsista_col_name].astype(str).replace({'1': 'Sim', '0': 'Não', 'True': 'Sim', 'False': 'Não'})

    # Map 'ATINGIU_PV' values to 'Atingiram' or 'Não Atingiram'
    pv_mapping = {
        'AVANÇO': 'Atingiram',
        'MANTIDO': 'Atingiram',
        'QUEDA': 'Não Atingiram',
        'INGRESSANTE': 'Não Atingiram'
    }
    df_filtered['pv_status'] = df_filtered[ponto_virada_col_name].astype(str).map(pv_mapping)

    # Drop rows where 'pv_status' is NaN (i.e., unmapped values from ATINGIU_PV)
    df_filtered.dropna(subset=['pv_status'], inplace=True)

    # Group by bolsista status and the new pv_status, then unstack
    ponto_virada_2022 = df_filtered.groupby(bolsista_col_name)['pv_status'].value_counts(normalize=True).unstack(fill_value=0)

    # Ensure both 'Atingiram' and 'Não Atingiram' columns exist, fill with 0 if missing
    for col in ['Atingiram', 'Não Atingiram']:
        if col not in ponto_virada_2022.columns:
            ponto_virada_2022[col] = 0

    # Manter os valores do eixo Y em porcentagem para exibição, sem impactar o hover
    ponto_virada_2022_for_text = ponto_virada_2022 * 100

    # Criar gráfico de barras para visualizar os resultados
    fig = go.Figure()

    # Adicionando as barras para "Não Bolsistas"
    if 'Não' in ponto_virada_2022.index:
        fig.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=[ponto_virada_2022.loc['Não', 'Não Atingiram'], ponto_virada_2022.loc['Não', 'Atingiram']],
            name='Não Bolsistas',
            marker_color='blue',
            text=[f"{ponto_virada_2022_for_text.loc['Não', 'Não Atingiram']:.2f}%", f"{ponto_virada_2022_for_text.loc['Não', 'Atingiram']:.2f}%"],
            textposition='inside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Adicionando as barras para "Bolsistas"
    if 'Sim' in ponto_virada_2022.index:
        fig.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=[ponto_virada_2022.loc['Sim', 'Não Atingiram'], ponto_virada_2022.loc['Sim', 'Atingiram']],
            name='Bolsistas',
            marker_color='green',
            text=[f"{ponto_virada_2022_for_text.loc['Sim', 'Não Atingiram']:.2f}%", f"{ponto_virada_2022_for_text.loc['Sim', 'Atingiram']:.2f}%"],
            textposition='inside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Ajustar o layout do gráfico
    fig.update_layout(
        title='Análise de Pontos de Virada entre Bolsistas e Não Bolsistas (2022)',
        xaxis_title='Ponto de Virada',
        yaxis_title='Porcentagem de Alunos (%)',
        yaxis_range=[0, 1],  # Os valores de 'y' são normalizados (entre 0 e 1)
        yaxis_tickformat='.0%',  # Exibir o eixo Y corretamente como porcentagem sem afetar o hover
        barmode='group',  # Agrupar barras lado a lado
        template='plotly'
    )

    # Exibir o gráfico
    fig.show()

In [ ]:
# Dynamically determine the correct column names for 'BOLSISTA' and 'PONTO_VIRADA' in df_2023
bolsista_col_name_2023 = get_column_name(df_2023, 'Indicado', '2023') # Use 'Indicado' as the base name
ponto_virada_col_name_2023 = get_column_name(df_2023, 'ATINGIU_PV', '2023') # Use 'ATINGIU_PV' as the base name

if bolsista_col_name_2023 is None:
    print(f"Error: 'BOLSISTA' (or 'Indicado') column not found in df_2023. Available columns: {df_2023.columns.tolist()}")
elif ponto_virada_col_name_2023 is None:
    print(f"Error: 'PONTO_VIRADA' (or 'ATINGIU_PV') column not found in df_2023. Available columns: {df_2023.columns.tolist()}")
else:
    print(f"\n--- Initial df_2023 Inspection ---")
    print(f"df_2023 columns: {df_2023.columns.tolist()}")

    if ponto_virada_col_name_2023 in df_2023.columns:
        print(f"df_2023['{ponto_virada_col_name_2023}'] head:\n{df_2023[ponto_virada_col_name_2023].head()}")
        print(f"df_2023['{ponto_virada_col_name_2023}'] value_counts (including NaN):\n{df_2023[ponto_virada_col_name_2023].value_counts(dropna=False)}")
    else:
        print(f"Column '{ponto_virada_col_name_2023}' not found in df_2023 after all. This should not happen if the earlier check passed.")

    if bolsista_col_name_2023 in df_2023.columns:
        print(f"df_2023['{bolsista_col_name_2023}'] head:\n{df_2023[bolsista_col_name_2023].head()}")
        print(f"df_2023['{bolsista_col_name_2023}'] value_counts (including NaN):\n{df_2023[bolsista_col_name_2023].value_counts(dropna=False)}")
    else:
        print(f"Column '{bolsista_col_name_2023}' not found in df_2023 after all. This should not happen if the earlier check passed.")

    # Contagem de alunos que atingiram ou não o Ponto de Virada por categoria (bolsistas ou não bolsistas)
    # Start with a copy of relevant columns only
    df_filtered_2023 = df_2023[[bolsista_col_name_2023, ponto_virada_col_name_2023]].copy()

    print(f"Shape of df_filtered_2023 before dropping NaNs in '{bolsista_col_name_2023}': {df_filtered_2023.shape}")
    print(f"NaNs in '{bolsista_col_name_2023}' before dropna: {df_filtered_2023[bolsista_col_name_2023].isnull().sum()}")
    # Drop rows where 'Indicado' column is NaN, as it's essential for grouping
    df_filtered_2023.dropna(subset=[bolsista_col_name_2023], inplace=True)
    print(f"Shape of df_filtered_2023 after dropping NaNs in '{bolsista_col_name_2023}': {df_filtered_2023.shape}")

    df_filtered_2023[bolsista_col_name_2023] = df_filtered_2023[bolsista_col_name_2023].astype(str).replace({'1': 'Sim', '0': 'Não', 'True': 'Sim', 'False': 'Não'})

    # Map 'ATINGIU_PV' values to 'Atingiram' or 'Não Atingiram'
    pv_mapping = {
        'AVANÇO': 'Atingiram',
        'MANTIDO': 'Atingiram',
        'QUEDA': 'Não Atingiram',
        'INGRESSANTE': 'Não Atingiram'
    }
    df_filtered_2023['pv_status'] = df_filtered_2023[ponto_virada_col_name_2023].astype(str).map(pv_mapping)

    print(f"\n--- Debugging 2023 Plot (after mapping) ---")
    print(f"Original '{ponto_virada_col_name_2023}' value counts:\n{df_filtered_2023[ponto_virada_col_name_2023].value_counts(dropna=False)}")
    print(f"'pv_status' value counts before final dropna:\n{df_filtered_2023['pv_status'].value_counts(dropna=False)}")
    print(f"DataFrame head after pv_status mapping:\n{df_filtered_2023.head()}")

    # Drop rows where 'pv_status' is NaN (i.e., unmapped values from ATINGIU_PV)
    df_filtered_2023.dropna(subset=['pv_status'], inplace=True)

    ponto_virada_2023 = df_filtered_2023.groupby(bolsista_col_name_2023)['pv_status'].value_counts(normalize=True).unstack(fill_value=0)

    for col in ['Atingiram', 'Não Atingiram']:
        if col not in ponto_virada_2023.columns:
            ponto_virada_2023[col] = 0

    print(f"Ponto Virada 2023 DataFrame before plotting:\n{ponto_virada_2023}")

    # Manter os valores do eixo Y em porcentagem para exibição, sem impactar o hover
    ponto_virada_2023_for_text = ponto_virada_2023 * 100

    # Criar gráfico de barras para visualizar os resultados
    fig_2023 = go.Figure()

    # Adicionando as barras para "Não Bolsistas"
    if 'Não' in ponto_virada_2023.index:
        fig_2023.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=[ponto_virada_2023.loc['Não', 'Não Atingiram'], ponto_virada_2023.loc['Não', 'Atingiram']],
            name='Não Bolsistas',
            marker_color='blue',
            text=[f"{ponto_virada_2023_for_text.loc['Não', 'Não Atingiram']:.2f}%", f"{ponto_virada_2023_for_text.loc['Não', 'Atingiram']:.2f}%"],
            textposition='inside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Adicionando as barras para "Bolsistas"
    if 'Sim' in ponto_virada_2023.index:
        fig_2023.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=[ponto_virada_2023.loc['Sim', 'Não Atingiram'], ponto_virada_2023.loc['Sim', 'Atingiram']],
            name='Bolsistas',
            marker_color='green',
            text=[f"{ponto_virada_2023_for_text.loc['Sim', 'Não Atingiram']:.2f}%", f"{ponto_virada_2023_for_text.loc['Sim', 'Atingiram']:.2f}%"],
            textposition='inside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Ajustar o layout do gráfico
    fig_2023.update_layout(
        title='Análise de Pontos de Virada entre Bolsistas e Não Bolsistas (2023)',
        xaxis_title='Ponto de Virada',
        yaxis_title='Porcentagem de Alunos (%)',
        yaxis_range=[0, 1],
        yaxis_tickformat='.0%',
        barmode='group',
        template='plotly'
    )

    # Exibir o gráfico
    fig_2023.show()

In [ ]:
# Dynamically determine the correct column names for 'BOLSISTA' and 'PONTO_VIRADA' in df_2024
bolsista_col_name_2024 = get_column_name(df_2024, 'Indicado', '2024') # Assuming 'Indicado' is the relevant column
ponto_virada_col_name_2024 = get_column_name(df_2024, 'ATINGIU_PV', '2024') # Assuming 'ATINGIU_PV' is the relevant column

if bolsista_col_name_2024 is None:
    print(f"Error: 'BOLSISTA' (or 'Indicado') column not found in df_2024. Available columns: {df_2024.columns.tolist()}")
elif ponto_virada_col_name_2024 is None:
    print(f"Error: 'PONTO_VIRADA' (or 'ATINGIU_PV') column not found in df_2024. Available columns: {df_2024.columns.tolist()}")
else:
    # Contagem de alunos que atingiram ou não o Ponto de Virada por categoria (bolsistas ou não bolsistas)
    df_filtered_2024 = df_2024.dropna(subset=[bolsista_col_name_2024, ponto_virada_col_name_2024]).copy()

    df_filtered_2024[bolsista_col_name_2024] = df_filtered_2024[bolsista_col_name_2024].astype(str).replace({'1': 'Sim', '0': 'Não', 'True': 'Sim', 'False': 'Não'})
    df_filtered_2024[ponto_virada_col_name_2024] = df_filtered_2024[ponto_virada_col_name_2024].astype(str).replace({'1': 'Sim', '0': 'Não', 'True': 'Sim', 'False': 'Não'})

    ponto_virada_2024 = df_filtered_2024.groupby(bolsista_col_name_2024)[ponto_virada_col_name_2024].value_counts(normalize=True).unstack()

    # Manter os valores do eixo Y em porcentagem para exibição, sem impactar o hover
    ponto_virada_2024_for_text = ponto_virada_2024 * 100

    # Criar gráfico de barras para visualizar os resultados
    fig_2024 = go.Figure()

    # Adicionando as barras para "Não Bolsistas"
    if 'Não' in ponto_virada_2024.index:
        fig_2024.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=ponto_virada_2024.loc['Não'],
            name='Não Bolsistas',
            marker_color='blue',
            text=[f'{val:.2f}%' for val in ponto_virada_2024_for_text.loc['Não']],
            textposition='outside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Adicionando as barras para "Bolsistas"
    if 'Sim' in ponto_virada_2024.index:
        fig_2024.add_trace(go.Bar(
            x=['Não Atingiram', 'Atingiram'],
            y=ponto_virada_2024.loc['Sim'],
            name='Bolsistas',
            marker_color='green',
            text=[f'{val:.2f}%' for val in ponto_virada_2024_for_text.loc['Sim']],
            textposition='outside',
            hovertemplate='%{text}<extra></extra>'
        ))

    # Ajustar o layout do gráfico
    fig_2024.update_layout(
        title='Análise de Pontos de Virada entre Bolsistas e Não Bolsistas (2024)',
        xaxis_title='Ponto de Virada',
        yaxis_title='Porcentagem de Alunos (%)',
        yaxis_range=[0, 1],
        yaxis_tickformat='.0%',
        barmode='group',
        template='plotly'
    )

    # Exibir o gráfico
    fig_2024.show()

In [ ]:
!pip install streamlit
import streamlit as st
import plotly.graph_objs as go
import plotly.express as px
import pandas as pd
from utils.charts import (
    format_number,
    plot_histograma
)